## GPT Architecture

In [ ]:
import torch
import torch.nn as nn

GPT_CONFIG_124M={
  "vocab_size": 50257,     # Vocabulary size
  "context_length": 1024,  # Context length
  "emb_dim": 768,          # Embedding dimension
  "n_heads": 12,           # Number of attention heads
  "n_layers": 12,          # Number of transformers
  "drop_rate": 0.1,        # Dropout rate
  "qkv_bias": False        # Query-Key-Value bias
}

### Layer Normalization

In [ ]:
torch.manual_seed(123)
batch_example=torch.randn(2, 5)
layer=nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out=layer(batch_example)
print(out)

- If `keepdim=True` is not set, then the returned tensor will be a 2D vector instead of a 2x1 dimensional vector.

In [ ]:
mean=out.mean(dim=-1, keepdim=True)
variance=out.var(dim=-1, keepdim=True)
print(f"Mean: {mean}")
print(f"Variance: {variance}")

In [ ]:
out_norm=(out-mean)/torch.sqrt(variance)
mean_norm=out_norm.mean(dim=-1, keepdim=True)
var_norm=out_norm.var(dim=-1, keepdim=True)

print(f"Normalized layer outputs: {out_norm}")
print(f"Normalized mean: {mean_norm}")
print(f"Normalized variance: {var_norm}")

In [ ]:
torch.set_printoptions(sci_mode=False)

print(f"Mean: {mean_norm}")
print(f"Variance: {var_norm}")

- The `scale` and `shift` are two trainable parameters that the LLM automatically adjust during the training which could improve the model's performance on its training task.

- This allows model to learn appropriate scaling and shifting that best suit the data it is processing.

- `unbiased=False` - we divide by the number of inputs n in the variance formula. When it is `True`, we apply `Bessel's correction`, where we divide by n-1 instead of n in the denominator to adjust for bias in sample variance estimation.

In [ ]:
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps=1e-5
    self.scale=nn.Parameter(torch.ones(emb_dim))
    self.shift=nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean=x.mean(dim=-1, keepdim=True)
    var=x.var(dim=-1, keepdim=True, unbiased=False)
    # To prevent from zero division
    norm_x=(x-mean)/torch.sqrt(var+self.eps)
    return self.scale*norm_x+self.shift

In [ ]:
ln=LayerNorm(emb_dim=5)
out_ln=ln(batch_example)
mean=out_ln.mean(dim=-1, keepdim=True)
var=out_ln.var(dim=-1, keepdim=True, unbiased=False)
print(f"Batch: {batch_example}")
print(f"Mean: {mean}")
print(f"Variance: {var}")

### GELU Activation

In [ ]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
  
  def forward(self, x):
    gelu=0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi, device=x.device))*(x+(0.044715*torch.pow(x, 3)))))
    return gelu

In [ ]:
import matplotlib.pyplot as plt

gelu, relu=GELU(), nn.ReLU()

x=torch.linspace(-3, 3, 100)
y_gelu, y_relu=gelu(x), relu(x)

plt.figure(figsize=(8, 3))
for i, (y, label) in enumerate(zip([y_gelu, y_relu], ["GELU", "RELU"]), 1):
  plt.subplot(1, 2, i)
  plt.plot(x, y)
  plt.title(f'{label} activation function')
  plt.xlabel('x')
  plt.ylabel(f'{label}(x)')
  plt.grid(True)

plt.tight_layout()
plt.show()

### Feed Forward Neural Network

- `Feed Forward` module is a small neural network consisting of 2 linear layers and a GELU activation function.

- Feed Forward module plays a crucial role in enhancing the model's ability to learn and generalize the data.

- Although the input and output dimensions of the module are same, it internally expands the embedding dimension into larger space in the first linear layer.

- This expansion is followed by a non-linear GELU activation and then contract back to the original dimension with second linear transformation.

- This allows for the exploration of a richer representation space and there is no need to worry about dimensional dispatch.

In [ ]:
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers=nn.Sequential(
      # Expansion
      nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
      # Activation
      GELU(),
      # Contraction
      nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
torch.manual_seed(123)

ffn=FeedForward(GPT_CONFIG_124M)
x=torch.rand(2, 3, 768)
out=ffn(x)
print(out.shape)

### Shortcut Connections

- `Shortcut Connections` are also known as skip or residual connections.

- Shortcut connections were first introduced in the field of Computer Vision to solve the problem of `vanishing gradients`.

- Shortcut connections create an alternative path for the gradient to flow, by skipping one or more layers. This is achieved by adding the output of one layer to the output of the latter layer.

- Since the loss function landscape becomes smooth, the gradient flow also becomes smooth.

In [ ]:
class ExampleShortcutNeuralNetwork(nn.Module):
  def __init__(self, layer_sizes, use_shortcut):
    super().__init__()
    self.use_shortcut=use_shortcut
    self.layers=nn.ModuleList([
      nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
      nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
      nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
      nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
      nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
    ])

  def forward(self, x):
    for layer in self.layers:
      layer_output=layer(x)

      if self.use_shortcut and x.shape==layer_output.shape:
        x=x+layer_output
      else:
        x=layer_output

    return x

In [ ]:
def print_gradients(model, x):
  # Forward Pass
  output=model(x)
  target=torch.tensor([[0.0]])

  # Calculate loss based on how close the target and output are
  loss=nn.MSELoss()
  loss=loss(output, target)

  # Backward Pass to calculate the gradients
  loss.backward()

  for name, param in model.named_parameters():
    if 'weight' in name:
      print(f'{name} has gradient mean of {param.grad.abs().mean().item()}')

#### 🔍 Code Overview

```python
def print_gradients(model, x):
```

This function accepts a **PyTorch model** and an input tensor `x`. Its purpose is to:

* Perform a **forward pass**
* Compute the **loss**
* Perform **backpropagation**
* Then print the **average magnitude of gradients** for each weight parameter

---

#### 🔢 1. **Forward Pass**

```python
  output = model(x)
```

* This feeds the input `x` into the model and gets its output.
* `output` is a tensor representing the model’s prediction for the given input.

---

#### 🎯 2. **Target Value**

```python
  target = torch.tensor([[0.0]])
```

* This is the **desired/true output** for the model (in this case, just 0.0).
* It is used to compute the **loss**, i.e., how far off the model’s prediction is from this value.

---

#### 📉 3. **Loss Calculation**

```python
  loss = nn.MSELoss()
  loss = loss(output, target)
```

* `nn.MSELoss()` is the **Mean Squared Error loss** — a standard regression loss.
* This measures the squared difference between `output` and `target`, i.e.:

  $$
  \text{loss} = \frac{1}{n} \sum (output_i - target_i)^2
  $$

---

#### 🔁 4. **Backward Pass**

```python
  loss.backward()
```

* This performs **backpropagation**, which:

  * Computes the **gradient of the loss** with respect to all **learnable parameters** in the model.
  * These gradients are stored in `.grad` attribute of each parameter (like `param.grad`).

---

#### 📊 5. **Printing Gradient Stats**

```python
  for name, param in model.named_parameters():
    if 'weight' in name:
      print(f'{name} has gradient mean of {param.grad.abs().mean().item()}')
```

* This loop goes through all parameters of the model.
* It filters out only the **weights** (ignoring biases).
* For each weight tensor:

  * It computes the **mean of the absolute gradient values**.
  * This tells how **strongly the loss is pushing on that layer’s weights**.
* `.item()` converts the scalar tensor to a Python float for clean printing.

---

In [ ]:
layer_sizes=[3, 3, 3, 3, 3, 1]
sample_input=torch.tensor([[1.0, 0.0, -1.0]])

In [ ]:
torch.manual_seed(123)

model_without_shortcut=ExampleShortcutNeuralNetwork(
  layer_sizes,
  use_shortcut=False
)
out=model_without_shortcut(sample_input)
print(out)

In [ ]:
print_gradients(model_without_shortcut, sample_input)

In [ ]:
torch.manual_seed(123)

model_with_shortcut=ExampleShortcutNeuralNetwork(
  layer_sizes,
  use_shortcut=True
)
out=model_with_shortcut(sample_input)
print(out)

In [ ]:
print_gradients(model_with_shortcut, sample_input)

### Coding a Transformer Block

#### Building Blocks of a Transformer

In [ ]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert (d_out%num_heads==0), \
    "d_out must be divisible by num_heads"

    self.d_out=d_out
    self.num_heads=num_heads
    self.head_dim=d_out//num_heads

    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_key=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out, bias=qkv_bias)
    # Linear layer to combine head outputs
    self.out_proj=nn.Linear(d_out, d_out)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in=x.shape

    queries=self.w_query(x)
    keys=self.w_key(x)
    values=self.w_value(x)

    # We implicitly split the matrix by adding a `num_heads` dimension
    # Unroll last dimension: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
    keys=keys.view(b, num_tokens, self.num_heads, self.head_dim)
    queries=queries.view(b, num_tokens, self.num_heads, self.head_dim)
    values=values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
    keys=keys.transpose(1, 2)
    queries=queries.transpose(1, 2)
    values=values.transpose(1, 2)

    # Computing attention scores
    attn_scores=torch.matmul(queries, keys.transpose(2, 3))

    # Original mask truncated to the number of tokens and converted to bool
    masked_bool=self.mask.bool()[:num_tokens, :num_tokens]

    # Using the mask to fill the attention scores
    masked_attn_scores=attn_scores.masked_fill_(masked_bool, -torch.inf)

    # Calculating the attention weights
    attn_weights=torch.softmax(masked_attn_scores/keys.shape[-1]**0.5, dim=-1)

    # Feeding attention weights to the dropout layer
    attn_weights=self.dropout(attn_weights)

    # Calculating context vectors
    context_vecs=(attn_weights@values).transpose(1, 2) # To get the original dimensions

    # Combining heads where self.d_out=num_heads*head_dim
    # contiguous - to make sure the reshaped matrices are in same blocks of memory
    context_vecs=context_vecs.contiguous().view(b, num_tokens, self.d_out)
    context_vecs=self.out_proj(context_vecs)

    return context_vecs

In [ ]:
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps=1e-5
    self.scale=nn.Parameter(torch.ones(emb_dim))
    self.shift=nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean=x.mean(dim=-1, keepdim=True)
    var=x.var(dim=-1, keepdim=True, unbiased=False)
    # To prevent from zero division
    norm_x=(x-mean)/torch.sqrt(var+self.eps)
    return self.scale*norm_x+self.shift

In [ ]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
  
  def forward(self, x):
    gelu=0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi, device=x.device))*(x+(0.044715*torch.pow(x, 3)))))
    return gelu

In [ ]:
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers=nn.Sequential(
      # Expansion
      nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
      # Activation
      GELU(),
      # Contraction
      nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
    )

  def forward(self, x):
    return self.layers(x)

#### Transformer Class

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att=MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_length"],
      dropout=cfg["drop_rate"],
      num_heads=cfg["n_heads"],
      qkv_bias=cfg["qkv_bias"]
    )
    self.ff=FeedForward(cfg)
    self.norm1=LayerNorm(cfg["emb_dim"])
    self.norm2=LayerNorm(cfg["emb_dim"])
    self.drop_shortcut=nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    # Shortcut connection for attention block
    shortcut=x
    # Every row in the input has 0 mean and 1 variance
    x=self.norm1(x)
    # We get the context vector of [batch_size, num_tokens, emb_dim]
    x=self.att(x)
    # Dropout layer to improve efficiency
    x=self.drop_shortcut(x)
    # Creating shortcut connection
    x=x+shortcut

    # Shortcut for feed forward block
    shortcut=x
    x=self.norm2(x)
    x=self.ff(x)
    x=self.drop_shortcut(x)
    x=x+shortcut

    return x

In [ ]:
torch.manual_seed(123)

x=torch.rand(2, 4, 768)
block=TransformerBlock(GPT_CONFIG_124M)
out=block(x)
print(f"Input shape: {out.shape}")
print(f"Output shape: {out.shape}")

- `Output` obtained is a context vector that encapsulates information from the entire input sequence i.e; the physical dimensions of the sequence remain unchanged as it passes through the transformer block, the content of each output vector is re-encoded to integrate contextual information from across the entire input sequence.

### Complete GPT Architecture

In [ ]:
import torch
import torch.nn as nn

class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.tok_emb=nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb=nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb=nn.Dropout(cfg["drop_rate"])

    self.trf_blocks=nn.Sequential(
      *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )

    self.final_norm=LayerNorm(cfg["emb_dim"])
    self.out_head=nn.Linear(
      cfg["emb_dim"], cfg["vocab_size"], bias=False
    )

  def forward(self, in_idx):
    batch_size, seq_len=in_idx.shape
    tok_embeds=self.tok_emb(in_idx)
    pos_embeds=self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x=tok_embeds+pos_embeds
    x=self.drop_emb(x)
    x=self.trf_blocks(x)
    x=self.final_norm(x)
    logits=self.out_head(x)
    return logits

- `Input dimension`: num_tokens x emb_dim

- `Output dimension`: num_tokens x vocab_size

### Step 1: Tokenization

In [ ]:
import tiktoken

tokenizer=tiktoken.get_encoding('gpt2')
batch=[]
txt1="Every effort moves you"
txt2="Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch=torch.stack(batch, dim=0)
print(batch)

### Step 2: Create an Instance of GPT Model

In [ ]:
torch.manual_seed(123)

model=GPTModel(GPT_CONFIG_124M)
logits=model(batch)
print(logits.shape)
print(logits)

In [ ]:
# Number of parameters

total_params=sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params}")

- We spoke of initializing a 124M parameter GPT model, but why is the actual number of parameters are 163M?

- The reason is a concept called `weight typing` that is used in original GPT-2 architecture, which means original GPT-2 architecture is reusing the weights from the token embedding layer in its output layer.

In [ ]:
print(f'Token Embedding Layer Shape: {model.tok_emb.weight.shape}')
print(f'Output Layer Shape: {model.out_head.weight.shape}')

In [ ]:
tot_params_gpt2=total_params-sum(p.numel() for p in model.out_head.parameters())
print(f'Number of trainable weights considering weight typing: {tot_params_gpt2}')

In [ ]:
tot_size_bytes=total_params*4
tot_size_mb=tot_size_bytes/(1024*1024)
print(f'Total size of the model: {tot_size_mb}')

### Generating Text from Output Tensors

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
  # idx is (batch_size, nu_tokens) array of indices in current context
  for _ in range(max_new_tokens):
    # Crop current context if it exceeds the supported context size
    '''For example, 
    Case 1: If LLM supports only 5 tokens and context size is
    10, then only the last 5 tokens are used as context.
    Case 2: If LLM supports 8 tokens and context size is 5, then
    only the last 5 tokens are used as context.'''
    idx_cond=idx[:, -context_size:]

    # Get output tensors - (batch_size, num_tokens, vocab_size)
    with torch.no_grad():
      logits=model(idx_cond)

    # Extract last vector
    logits=logits[:, -1, :]

    # Apply softmax to get probabilities - (batch_size, vocab_size)
    probs=torch.softmax(logits, dim=-1)

    # Get the idx of the vocab entry with the highest probability value
    idx_next=torch.argmax(probs, dim=-1, keepdim=True)
    # (batch_size, 1)

    # Append sampled index to the running sequence
    idx=torch.cat((idx, idx_next), dim=1)
    # (batch_size, num_tokens+1)

  return idx

In [ ]:
start_context="Hello, I am"
encoded_text=tokenizer.encode(start_context) # [4]
# Adds a new dimension at the specified position
encoded_tensor=torch.tensor(encoded_text).unsqueeze(0) # [1, 4]
print(f"Encoded Tensor: {encoded_tensor}")
print(f"Encoded Tensor Shape: {encoded_tensor.shape}")

In [ ]:
model.eval()

output=generate_text_simple(
  model=model,
  idx=encoded_tensor,
  max_new_tokens=6,
  context_size=GPT_CONFIG_124M["context_length"]
)

print(f"Output: {output}")
print(f"Output Shape: {output.shape}")

In [ ]:
decoded_text=tokenizer.decode(output.squeeze(0).tolist())
print(f"Decoded Text: {decoded_text}")